In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

import ipywidgets as widgets
from IPython.display import display

df = pd.read_csv("../data/raw/bach_chorales.dataset")

pitch_cols = df.columns[2:14]
df[pitch_cols] = (df[pitch_cols] == "YES") # Converts "YES" to 1 and "NO" to 0

In [2]:
NOTE = {"C":0,"D":2,"E":4,"F":5,"G":7,"A":9,"B":11}
ACC  = {"b":-1,"_":0,"#":1}

INV = {
    0: "root",
    3: "1st", 4: "1st",
    6: "2nd", 7: "2nd",
    10:"3rd", 11:"3rd",
}

def pc(note):
    return (NOTE[note[0]] + (ACC.get(note[1],0) if len(note)>1 else 0)) % 12

def inversion(row):
    label = row["chord_label"]

    root = (NOTE[label[0]] + ACC[label[1]]) % 12
    bass = pc(row["bass"])

    interval = (bass - root) % 12
    return INV.get(interval, "non-chord")

df["inversion"] = df.apply(inversion, axis=1)

In [3]:
def plot_piece(piece_id):
    piece = df[df["choral_ID"] == piece_id].sort_values("event_number")

    fig, axs = plt.subplots(3, 1, figsize=(10, 12))

    # --- A. Distribution ---
    counts = piece["inversion"].value_counts(normalize=True)
    axs[0].bar(counts.index, counts.values)
    axs[0].set_title("Inversion distribution")

    # --- B. Timeline ---
    axs[1].plot(piece["event_number"], piece["inversion"])
    axs[1].set_title("Inversion over time")
    axs[1].set_yticks([0,1,2,3])
    axs[1].set_yticklabels(["root","1st","2nd","3rd"])

    # --- C. Rolling proportions ---
    dummies = pd.get_dummies(piece["inversion"])
    rolling = dummies.rolling(window=10, min_periods=1).mean()
    rolling.plot(ax=axs[2])
    axs[2].set_title("Rolling inversion usage (window=10)")

    plt.tight_layout()
    plt.show()

In [4]:
dropdown = widgets.Dropdown(
    options=df["choral_ID"].unique(),
    description="Chorale:"
)

widgets.interactive(plot_piece, piece_id=dropdown)


interactive(children=(Dropdown(description='Chorale:', options=('000106b_', '000206b_', '000306b_', '000408b_'…